# 04: Run your circuit on a remote CPU

**What you will learn:** a cloud job runs elsewhere and has its own ID, status and result. The circuit is still the same FlagQuantum circuit.

We will send one Bell circuit to Jiuding. This is a CPU experiment, not an accelerator or speedup demonstration.


## 1. Prepare the workshop

Run this setup cell unchanged. It finds your personal result directory. The event environment supplies credentials and the job image; no secrets belong in notebook cells. See [Setup](../../SETUP.md) when using your own computer.


In [ ]:
from pathlib import Path
import os
import sys
import json

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next((root / "flagos2026" for root in search_roots
                 if (root / "flagos2026/notebooks").is_dir()), None)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks. Open the deployed workshop first.")
sys.path.insert(0, str(WORKSHOP / "scripts"))
from cloud_lesson import workshop_directory

WORKSHOP = workshop_directory(Path.cwd())
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)


## 2. Build a Bell circuit and predict the answer

The ideal probabilities for `00` and `11` are each one half. First check this locally.


In [ ]:
import flagquantum as fq

circuit = fq.Circuit(2).h(0).cx(0, 1)
local = fq.run(circuit)
print("Local probabilities:", local.state.abs().square().tolist())


## 3. Choose the remote CPU

The image supplies the installed software. The event environment preconfigures it; at home use your own Jiuding image, project and queue. **Enabling the switch and running the submission cell creates a real CPU job and consumes queue resources.**


In [ ]:
target = "jiuding:cpu"
image = os.getenv("JIUDING_IMAGE", "")
project = os.getenv("JIUDING_PROJECT", "")
queue = os.getenv("JIUDING_QUEUE")
submit_to_cloud = False


### Prepare recovery files (run unchanged)

This folded cell protects saved tasks from accidental duplicate submissions. It checks the circuit/configuration and restores a matching receipt with `fq.restore_job()`. It never submits work. Storage and credentials are setup concerns, not quantum operations.


In [ ]:
from flagquantum.core.ir import ensure_circuit_ir

receipt_path = OUTPUTS / "jiuding-bell-lesson.receipt.json"
guard_path = OUTPUTS / "jiuding-bell-lesson.submission.json"
request = {
    "circuit_hash": ensure_circuit_ir(circuit).content_hash,
    "target": target,
    "shots": None,
    "image": image,
    "project": project,
    "queue": queue,
}
job = None
if submit_to_cloud:
    if not image or not project:
        raise RuntimeError("Ask your instructor to configure the Jiuding image and project; nothing submitted.")
    if receipt_path.exists():
        saved = json.loads(guard_path.read_text())
        if any(saved.get(key) != value for key, value in request.items()):
            raise RuntimeError("Saved task belongs to another experiment. Keep its files and use a new receipt filename.")
        job = fq.restore_job(receipt_path)
    else:
        try:
            with guard_path.open("x") as stream:
                json.dump(request, stream, indent=2)
        except FileExistsError as exc:
            raise RuntimeError("An earlier attempt has no receipt. Reconcile it on the platform; do not resubmit.") from exc


## 4. Submit through FlagQuantum

`fq.submit()` returns without waiting for execution. Save its receipt immediately with `job.save()` so the same task can be restored after a kernel restart. Rerunning this notebook restores the recorded task instead of creating another one.


In [ ]:
if submit_to_cloud and job is None:
    job = fq.submit(
        circuit, target=target, image=image, project=project, queue=queue,
    )
    print("Job ID:", job.id)
    job.save(receipt_path)
elif job is not None:
    print("Restored job:", job.id)
else:
    print("Local rehearsal only. Enable the switch to submit a real task.")


## 5. Query progress

This is a single read-only query. `queued` is waiting, `running` is executing, and `succeeded` means the result can be retrieved.


In [ ]:
if job is not None:
    print("Status:", job.status())


## 6. Wait briefly and read the result

`job.wait()` is a convenience for polling status and retrieving the result. A timeout leaves the task running. Rerun this cell later. Alternatively use `job.result()` once the status is `succeeded`. No local result is substituted for a remote failure.


In [ ]:
result = None
if job is not None:
    try:
        result = job.wait(timeout=30)
    except TimeoutError:
        print("Still", job.raw_status, "— rerun this result cell later; do not resubmit.")
if result is not None:
    import torch

    torch.testing.assert_close(result.state, local.state)
    print("Remote probabilities:", result.state.abs().square().tolist())
    print("The remote CPU calculation matches the local calculation.")


## Take it home

The standard API is the same outside Jupyter. This short example creates a new real task each time it is deliberately invoked; configure your image, project and queue first.

```python
import os
import flagquantum as fq

circuit = fq.Circuit(2).h(0).cx(0, 1)
job = fq.submit(
    circuit, target="jiuding:cpu", image=os.environ["JIUDING_IMAGE"],
    project=os.environ["JIUDING_PROJECT"], queue=os.getenv("JIUDING_QUEUE"),
)
job.save("bell.receipt.json")
print(job.id, job.status())
```

For a saved task: `job = fq.restore_job("bell.receipt.json")`. When complete, inspect `job.result().state`.

**Explain the result:** which calls submit, query, wait and retrieve? Does remote execution of a tiny circuit prove a speedup? Use [Cloud job field notes](../../CLOUD_JOBS.md) for complete Python programs and accelerator choices.
